## Connecting Snowflake


In [281]:

import snowflake.connector as connector
import pandas as pd
import numpy as np


# Conection details
connection_details = {
    'user': 'NASRULKHAIR',
    'password': 'C@de0123456789',
    'account': 'SSDSCZP-MD39179',
    'warehouse': 'PROJECT_WH',
    'database': 'ECOMMERCE_PROJECT',
    'schema': 'RAW'
}

# create connection
conn = connector.connect(**connection_details)
cur = conn.cursor()



## Querying the database

In [282]:
query = "SELECT * FROM ECOMMERCE_PROJECT.RAW.ETSY;"

df = pd.read_sql(query, conn)
print(df.head())

C:\Users\User\AppData\Local\Temp\ipykernel_26144\1415552792.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


                                                 URL  PRODUCT_ID  \
0  https://www.etsy.com/listing/1001239907/boho-w...  1001239907   
1  https://www.etsy.com/listing/195137531/m-brida...   195137531   
2  https://www.etsy.com/listing/1336040522/acryli...  1336040522   
3  https://www.etsy.com/listing/787484799/infinit...   787484799   
4  https://www.etsy.com/listing/609456265/wedding...   609456265   

                                               TITLE RATING  \
0  Boho Wedding Guest Book. Gold Foil Wedding Gue...    4.9   
1  M bridal earrings jewelry bridesmaid gift wedd...    4.8   
2  Acrylic Married Ornament Gift Newlywed Gift Mr...      5   
3  Infinite Love Personalized Wedding Frame, Wedd...    4.9   
4  Wedding Poems,Marriage Takes Three,Custom Wedd...      5   

   REVIEWS_COUNT_SHOP  REVIEWS_COUNT_ITEM  INITIAL_PRICE DISCOUNT_PERCENTAGE  \
0                6459                  11          28.00                   0   
1               11611                  13          3

In [283]:
df.columns

Index(['URL', 'PRODUCT_ID', 'TITLE', 'RATING', 'REVIEWS_COUNT_SHOP',
       'REVIEWS_COUNT_ITEM', 'INITIAL_PRICE', 'DISCOUNT_PERCENTAGE',
       'FINAL_PRICE', 'CURRENCY', 'IMAGES', 'BREADCRUMBS', 'ROOT_CATEGORY',
       'SELLER_NAME', 'SELLER_SHOP_NAME', 'SELLER_RESPONSE', 'ITEM_DETAILS',
       'SHIPPING_RETURN_POLICIES', 'PRODUCT_SPECIFICATIONS',
       'RELATED_SEARCHES', 'FAQS', 'CATEGORY_TREE', 'TOP_REVIEWS',
       'PHOTOS_FROM_REVIEWS', 'LIISTED_DATE', 'TIMESTAMP'],
      dtype='object')

In [284]:
df.shape

(1000, 26)

## Cleaning & Transformation

##### COL: CATEGORY_THREE

In [285]:
df["CATEGORY_TREE"].head()

0    All categories,Weddings,Gifts & Mementos,Guest...
1             All categories,Weddings,Jewelry,Earrings
2    All categories,Home & Living,Home Decor,Season...
3    All categories,Weddings,Gifts & Mementos,Portr...
4    All categories,Weddings,Gifts & Mementos,Gifts...
Name: CATEGORY_TREE, dtype: object

In [286]:
# handling hierachical data
df_split = df["CATEGORY_TREE"].str.split(',', expand=True)
df_split = df_split.replace('All categories', pd.NA)

def shift_left(row):
    vals = [x for x in row if pd.notna(x)]
    return vals + [pd.NA] * (len(row)   - len(vals))

df_split = df_split.apply(shift_left, axis=1, result_type='expand')

# keep only the first 3 level
df_split = df_split.iloc[:, :3]
df_split.columns = ["Root", "Leaf", "Sub_leaf"]

# merge into the raw table
df = pd.concat([df.drop(columns=['CATEGORY_TREE']), df_split], axis=1)
print(df.head())


                                                 URL  PRODUCT_ID  \
0  https://www.etsy.com/listing/1001239907/boho-w...  1001239907   
1  https://www.etsy.com/listing/195137531/m-brida...   195137531   
2  https://www.etsy.com/listing/1336040522/acryli...  1336040522   
3  https://www.etsy.com/listing/787484799/infinit...   787484799   
4  https://www.etsy.com/listing/609456265/wedding...   609456265   

                                               TITLE RATING  \
0  Boho Wedding Guest Book. Gold Foil Wedding Gue...    4.9   
1  M bridal earrings jewelry bridesmaid gift wedd...    4.8   
2  Acrylic Married Ornament Gift Newlywed Gift Mr...      5   
3  Infinite Love Personalized Wedding Frame, Wedd...    4.9   
4  Wedding Poems,Marriage Takes Three,Custom Wedd...      5   

   REVIEWS_COUNT_SHOP  REVIEWS_COUNT_ITEM  INITIAL_PRICE DISCOUNT_PERCENTAGE  \
0                6459                  11          28.00                   0   
1               11611                  13          3

##### COL : TITLE

In [287]:
df['TITLE'].head()

df['TITLE']= df["TITLE"].str.split(r'[.,]', expand=True)[0]
df['TITLE'].value_counts()




TITLE
Father of the Bride Gift                                                                                                                      3
Friend gift friendship gift ideas Friends that are family wooden plaque bff gift best friend christmas gift AM28                              3
Playhouse Build Plans for Kids                                                                                                                3
Personalised gift for your best friend - choose your quote                                                                                    3
Baby Shower Gift                                                                                                                              3
                                                                                                                                             ..
The Creator (Custom Soundtrack Cover) by Hans Zimmer (Original Motion Picture Soundtrack)                                         

##### COL: PRODUCT_SPECIFICATIONS

In [288]:
# handling PRODUCT_SPECIFICATIONS
import pandas as pd
import json

# 1. Safely convert JSON-like strings to Python objects
def safe_json_load(x):
    try:
        return json.loads(x.replace("'", '"'))  # replace single quotes with double quotes
    except (json.JSONDecodeError, AttributeError):
        return []  # return empty list if parsing fails

df['PRODUCT_SPECIFICATIONS'] = df['PRODUCT_SPECIFICATIONS'].apply(safe_json_load)

# 2. Extract only 'specification_name' values
def get_spec_names(spec_list):
    return [spec.get('specification_name') for spec in spec_list if spec.get('specification_name')]

df['SPEC_NAMES'] = df['PRODUCT_SPECIFICATIONS'].apply(get_spec_names)

# 3. Join multiple names into a single string per row (optional)
df['SPEC_NAMES'] = df['SPEC_NAMES'].apply(lambda x: ', '.join(x))

# 4. Drop the original column
df = df.drop(columns=['PRODUCT_SPECIFICATIONS'])

# 5. Inspect the result
print(df.head())




                                                 URL  PRODUCT_ID  \
0  https://www.etsy.com/listing/1001239907/boho-w...  1001239907   
1  https://www.etsy.com/listing/195137531/m-brida...   195137531   
2  https://www.etsy.com/listing/1336040522/acryli...  1336040522   
3  https://www.etsy.com/listing/787484799/infinit...   787484799   
4  https://www.etsy.com/listing/609456265/wedding...   609456265   

                                               TITLE RATING  \
0                            Boho Wedding Guest Book    4.9   
1  M bridal earrings jewelry bridesmaid gift wedd...    4.8   
2  Acrylic Married Ornament Gift Newlywed Gift Mr...      5   
3           Infinite Love Personalized Wedding Frame    4.9   
4                                      Wedding Poems      5   

   REVIEWS_COUNT_SHOP  REVIEWS_COUNT_ITEM  INITIAL_PRICE DISCOUNT_PERCENTAGE  \
0                6459                  11          28.00                   0   
1               11611                  13          3

## Data Normalization

**Dimension Tables**

<u>dim_product</u>

- PK: ProductID (surrogate key)
- Attributes: ProductCode (from initial PRODUCT_ID), Title, ItemDetails, ProductSpecifications, Images, CategoryID(FK),

<u>dim_category</u>

- PK: CategoryID
- Attributes: RootCategory, CategoryTree, Breadcrumbs

<u>dim_seller</u>

- PK: SellerID
- Attributes: SellerName, SellerShopName, SellerResponse



**Fact Table**

Since this dataset looks like a product catalog with availability, a natural fact table would be FactProductAvailability.

<u>fact_pricing</u>

- PK: Composite (DateID, ProductID, DomainID)
- Measures: InitialPrice, DiscountPercentage, FinalPrice, Currency.

<u>fact_pricing</u>

- PK: Composite (ProductID)
- Measures: Rating, ReviewsCountShop, ReviewsCountItem

#### Dimension Tables

In [289]:
# create function for reproducible

def create_dim_table(df, columns, id_name, subset_col=None):
    dim = df[columns].drop_duplicates(subset = subset_col).reset_index(drop=True)
    dim.insert(0, id_name, range(1, len(dim) + 1))
    return dim



In [295]:
# dim_product
dim_product = create_dim_table(df, ["PRODUCT_ID", "TITLE", "ITEM_DETAILS", "SPEC_NAMES", "IMAGES"], "ProductID")
dim_product.rename(columns={'PRODUCT_ID': 'Product_Code'},inplace=True)
dim_product.columns = dim_product.columns.str.strip().str.lower()
dim_product.head()



,productid,product_code,title,item_details,spec_names,images
0,1,1001239907,Boho Wedding Guest Book,Welcome to Drift House Press ~ Please check ou...,"Book Size | Page Count + Type, Upgrade To Foil...",https://i.etsystatic.com/14608740/r/il/8043d6/...
1,2,195137531,M bridal earrings jewelry bridesmaid gift wedd...,Tarnish resistant White gold plated metal fram...,Finish,https://i.etsystatic.com/5760764/r/il/3f413c/6...
2,3,1336040522,Acrylic Married Ornament Gift Newlywed Gift Mr...,Make your family memories bright with Hello De...,Gift Box Option,https://i.etsystatic.com/7634662/r/il/2e49ae/4...
3,4,787484799,Infinite Love Personalized Wedding Frame,The Infinite Love Personalized Wedding Frame i...,"Size, Title + Text Color",https://i.etsystatic.com/11355547/r/il/b8fd5e/...
4,5,609456265,Wedding Poems,Personalized Wedding Gift - Marriage Takes Thr...,Newsletter,https://i.etsystatic.com/16815897/r/il/1b9cc7/...


In [291]:
# dim_category
dim_category = create_dim_table(df, ['Root', 'Leaf', 'Sub_leaf'], 'CategoryID')
dim_category.columns = dim_category.columns.str.strip().str.lower()
dim_category.value_counts()

categoryid  root                       leaf                     sub_leaf            
1           Weddings                   Gifts & Mementos         Guest Books             1
2           Weddings                   Jewelry                  Earrings                1
3           Home & Living              Home Decor               Seasonal Decor          1
4           Weddings                   Gifts & Mementos         Portraits & Frames      1
5           Weddings                   Gifts & Mementos         Gifts For The Couple    1
                                                                                       ..
148         Weddings                   Accessories              Bags & Purses           1
150         Jewelry                    Earrings                 Hoop Earrings           1
151         Weddings                   Clothing                 Dresses                 1
152         Home & Living              Lighting                 Ceiling Fans            1
153         Ele

In [296]:
# dim_seller
dim_seller = create_dim_table(df,['SELLER_NAME', 'SELLER_SHOP_NAME', 'SELLER_RESPONSE'], 'SellerID')
dim_seller.columns = dim_seller.columns.str.strip().str.lower()
dim_seller['seller_shop_name'].value_counts()

# hadling duplicated shopname by dropping subset method
dim_seller = dim_seller.drop_duplicates(subset=['seller_shop_name'], keep='first').reset_index(drop=True)
dim_seller['seller_shop_name'].value_counts()



seller_shop_name
MydfDesignShirt        1
DriftHousePress        1
DesignByKara           1
HelloDelicious         1
PersonalizationMall    1
                      ..
EngraveMyMemories      1
MilestoneGiftShoppe    1
LuckandLuck            1
GuzmanCreations        1
Tumblerfy              1
Name: count, Length: 870, dtype: int64